# Mnist Qiskit: Amplitude encoding with 10 qubits, quantum neural network
This is a draft.
Using all samples from the MNIST wouldn't be feasable, so, the question is:
What's the perfomance of this model against a classical model (both classical machine learning and some sort of neural network)

In [3]:
from sklearn.datasets import fetch_openml
mnist = fetch_openml('mnist_784', version=1)
X, y = mnist.data, mnist.target
y = mnist.target.astype(int)

In [4]:
import numpy as np
from scipy.optimize import minimize
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector
from qiskit.circuit import ParameterVector

# We need to:
# normalize each image vector;
# pad it to length 2^n (mnist has length of 784, means 10 qubit (2^10 = 1024) is enough) 
# use qc.initialize(state_vector, qubits), where state_vector is the padded image to 1024 and qubits is the range(10).
def amplitude_encode(image):
    """
    where x is the image matrice
        flatten: m x n => m * n
        normalize: x => x / ||x||
        pad to 10 qubits: np.zeros(1024) then rewrite the first 784 with the data from x 
    """
    x = np.asarray(image, dtype=float).flatten()

    norm = np.linalg.norm(x)
    if norm == 0:
        raise ValueError("Zero vector")

    x = x / norm

    padded = np.zeros(1024)
    padded[:len(x)] = x

    qc = QuantumCircuit(10)
    qc.initialize(padded, range(10))
    return qc


# Simple variational model
def build_model(qc, n_layers=1):
    """
        The model consists of:
        We want to do a rotation on the y axis on each qubit then
        do a control x-gate to ensure entaglement, which means
        if q = |0> then q + 1 = q + 1, stay the same
        else (q = |1>) then q + 1 = |0> (if it was |1>) or q + 1 = |1> (if it was |0>)
    """
    params = ParameterVector("θ", length=10 * n_layers)

    idx = 0
    for _ in range(n_layers):
        for q in range(10):
            qc.ry(params[idx], q)
            idx += 1

        for q in range(9):
            qc.cx(q, q + 1)

    return qc, params


def forward_pass(qc, param_values):
    # Bind parameters
    bound = qc.assign_parameters(param_values, inplace=False)
    # Get statevector
    state = Statevector.from_instruction(bound).data
    # Probabilities of all 1024 basis states
    probs = np.abs(state) ** 2
    # Map amplitudes to a signal in [-1, 1]
    # For example: sum over all states weighted by (-1)^number_of_ones
    pred = 0
    for i, p in enumerate(probs):
        # Count number of ones in the binary representation
        n_ones = bin(i).count("1")
        pred += ((-1) ** n_ones) * p
    return np.real(pred)

# DATA (30 samples)
X = np.asarray(X, dtype=float)
y = np.asarray(y, dtype=int)

# binary labels for quantum model stability
y = 2 * y - 1  # {-1, +1}

circuits = []
labels = []

for i in range(30):
    qc = amplitude_encode(X[i])
    qc, _ = build_model(qc, n_layers=1)
    circuits.append(qc)
    labels.append(y[i])

# Shared parameters
n_params = 10 * 1
init_params = np.random.rand(n_params)

# Batch loss function
def loss_fn(params):
    total = 0.0
    for qc, label in zip(circuits, labels):
        pred = forward_pass(qc, params)
        total += (pred - label) ** 2
    return total / len(circuits)

# Optimization
result = minimize(
    loss_fn,
    init_params,
    method="COBYLA"
)

trained_params = result.x

print("Final loss:", result.fun)

Final loss: 70.70432291845962
